# Stage 5 — Momentum Overlay

Layer a cross-sectional **momentum** signal onto the carry sort and ask the §11
question, made falsifiable: does momentum reduce MaxDD / CVaR99 at **less** Sharpe
cost than the Stage-3 hedges did? The bar to beat is per-currency RR (combined net
Sharpe 0.47 -> 0.457, MaxDD ~ -28%, skew -0.65 -> -0.60).

Momentum = trailing cumulative **excess** return (`fx.momentum_panel`), lookback grid
**21 / 63 / 252 d** (1/3/12M; Burnside et al. 2011, Menkhoff et al. 2012b). Four
strategy families, each reusing `carry_portfolio`:

| family | signal fed to `carry_portfolio` |
|---|---|
| **pure carry** (baseline) | `carry` |
| **(a) pure momentum** | `momentum_panel(xret, L)` |
| **(b) double-sort filter** | `carry`, `filter_signal=mom_L` (long high-carry & mom>=0, short low-carry & mom<=0) |
| **(c) blend** | `0.5*z(carry) + 0.5*z(mom_L)` |

All tracks are vol-targeted (10% / 60d, the §6.6 sizing standard) so Sharpe/tail
numbers are comparable to the Stage-3 0.457 benchmark. Every track is reported
**gross AND net** (`forward_halfspreads` + `roundtrip_cost`), both universes
(G10 tercile / ALL quintile), common window **2007-05 -> 2026-06**, IR vs DBHVG10U
(G10) / FXCTEM8 (combined).

In [1]:
# ===== 0. Setup =====
from pathlib import Path
import numpy as np, pandas as pd
import fx_utils as fx

OUT = Path('outputs')
g10_px = fx.load_wide('g10_fx_spot_forward')
em_px  = fx.load_wide('em_fx_spot_forward')
spots  = fx.spots_usd_per_fx(g10_px, em_px)
carry  = fx.carry_panel(g10_px, em_px, tenor='1M')
xret   = fx.excess_returns(spots, carry)

UNIVERSE_G10 = ['AUD','CAD','CHF','EUR','GBP','JPY','NOK','NZD','SEK']
UNIVERSE_ALL = [c for c in xret.columns if c not in ('HKD','DKK','CNY')]
U         = {'G10': UNIVERSE_G10, 'ALL': UNIVERSE_ALL}
N_BUCKETS = {'G10': 3, 'ALL': 5}
BENCH     = {'G10': 'DBHVG10U', 'ALL': 'FXCTEM8'}
LOOKBACKS = [21, 63, 252]
WINDOW    = slice('2007-05-01', '2026-06-30')

bmk = fx.benchmark_returns()
hs_out, hs_pts = fx.forward_halfspreads('1M')
print('xret', xret.shape, '| G10', len(UNIVERSE_G10), '| ALL', len(UNIVERSE_ALL))
print('benchmarks:', list(bmk.columns))

xret (5094, 29) | G10 9 | ALL 27
benchmarks: ['DBHVG10U', 'FXCTEM8', 'DBHVBUSI']


## 1. Signals and weight panels

`momentum_panel` returns a daily trailing-cumulative-excess-return panel;
`carry_portfolio` then samples it month-end and shifts one day, exactly as it does
for carry (no lookahead). The blend z-scores each signal **within the traded
cross-section** so carry (annualised) and momentum (cumulative return) are on a
common scale before the 50/50 combination.

In [2]:
# ===== 1. Signals + vol-targeted weights =====
mom = {L: fx.momentum_panel(xret, L) for L in LOOKBACKS}

def vt(w):
    return fx.vol_target_weights(w, xret)

def blend_signal(un, L):
    cols = carry.columns.intersection(mom[L].columns).intersection(un)
    return 0.5 * fx.zscore_xs(carry[cols]) + 0.5 * fx.zscore_xs(mom[L][cols])

weights = {}
for uni in ('G10', 'ALL'):
    nb, un = N_BUCKETS[uni], U[uni]
    weights[(uni, 'carry')] = vt(fx.carry_portfolio(carry, xret, n_buckets=nb, universe=un))
    for L in LOOKBACKS:
        weights[(uni, f'mom{L}')]    = vt(fx.carry_portfolio(mom[L], xret, n_buckets=nb, universe=un))
        weights[(uni, f'filter{L}')] = vt(fx.carry_portfolio(carry, xret, n_buckets=nb, universe=un,
                                                             filter_signal=mom[L]))
        weights[(uni, f'blend{L}')]  = vt(fx.carry_portfolio(blend_signal(un, L), xret,
                                                             n_buckets=nb, universe=un))
print(len(weights), 'weight panels:',
      sorted({k[1] for k in weights}))

20 weight panels: ['blend21', 'blend252', 'blend63', 'carry', 'filter21', 'filter252', 'filter63', 'mom21', 'mom252', 'mom63']


## 2. Tracks — gross and net of costs

Gross = `portfolio_returns`; net = gross - `roundtrip_cost(w, hs_out, hs_pts)`
(new notional pays the outright half-spread, maintained notional rolls at the points
half-spread). The vol-target scalar changes notional daily, so the filter/blend
books trade a little more than pure carry — priced here, not assumed away.

In [3]:
# ===== 2. Gross / net tracks =====
tracks = {}
for (uni, strat), w in weights.items():
    g = fx.portfolio_returns(w, xret).rename(f'{uni}_{strat}_gross')
    c = fx.roundtrip_cost(w, hs_out, hs_pts)
    n = (g - c).rename(f'{uni}_{strat}_net')
    tracks[g.name] = g
    tracks[n.name] = n

daily = pd.concat(list(tracks.values()) + [bmk['DBHVG10U'], bmk['FXCTEM8']], axis=1).loc[WINDOW]
print('daily panel', daily.shape, '| tracks', len(tracks))

daily panel (5008, 42) | tracks 40


## 3. Statistics — full metrics, IR, turnover, cost drag, NW alpha vs carry

IR from `summary_stats(benchmark=...)` (G10 -> DBHVG10U, ALL -> FXCTEM8). NW alpha
regresses each overlay on the **same-basis pure-carry** track: a positive, robust
`t_alpha_vs_carry` would mean momentum adds return; a flat one with a better tail
means it only reshapes risk.

In [4]:
# ===== 3. Stats table =====
g10_cols = [c for c in daily.columns if c.startswith('G10_')]
all_cols = [c for c in daily.columns if c.startswith('ALL_')]
summary = pd.concat([
    fx.summary_stats(daily[g10_cols], benchmark=bmk['DBHVG10U']),
    fx.summary_stats(daily[all_cols], benchmark=bmk['FXCTEM8']),
    fx.summary_stats(daily[['DBHVG10U', 'FXCTEM8']]),
])

# turnover (weight-based; same for gross/net), cost drag (net), NW alpha vs carry
meta = {}
for (uni, strat), w in weights.items():
    to = fx.turnover(w)
    c  = fx.roundtrip_cost(w, hs_out, hs_pts)
    live = w.abs().sum(axis=1) > 0
    drag = float(c[live].loc[WINDOW].mean() * fx.ANN_DAYS)
    for basis in ('gross', 'net'):
        key = f'{uni}_{strat}_{basis}'
        m = meta.setdefault(key, {})
        m['avg_monthly_turnover'] = to
        if basis == 'net':
            m['ann_cost_drag'] = drag
        if strat != 'carry':
            base = daily[f'{uni}_carry_{basis}']
            res = fx.nw_regression(daily[key], base.to_frame('carry'), lags=5)
            m['base'] = f'{uni}_carry_{basis}'
            m['alpha_vs_carry_ann'] = res['alpha_ann']
            m['t_alpha_vs_carry']   = res['alpha_t']

summary = summary.join(pd.DataFrame(meta).T)
print(summary.shape, 'rows')
summary[['sharpe', 'max_drawdown', 'CVaR_99', 'skew', 'info_ratio']].round(3)

(42, 27) rows


,sharpe,max_drawdown,CVaR_99,skew,info_ratio
G10_carry_gross,0.166879,-0.365337,0.031562,-0.953797,0.269403
G10_carry_net,0.119133,-0.382324,0.031584,-0.952499,0.214478
G10_mom21_gross,0.089169,-0.410232,0.023615,0.154773,0.118944
G10_mom21_net,-0.16036,-0.579087,0.023863,0.134934,-0.067609
G10_filter21_gross,0.011752,-0.436715,0.026532,-0.41857,0.072902
G10_filter21_net,-0.106819,-0.522131,0.026612,-0.415812,-0.03267
G10_blend21_gross,0.039096,-0.309377,0.028483,-1.336107,0.103081
G10_blend21_net,-0.149287,-0.495118,0.02878,-1.329737,-0.076266
G10_mom63_gross,-0.180256,-0.597988,0.0262,1.381237,-0.087958
G10_mom63_net,-0.341499,-0.710698,0.026415,1.367988,-0.212001


## 4. Track correlation matrix

Family (a)'s premise: is momentum diversifying (low/negative correlation to carry)?
The carry<->momentum cells of the net-track correlation matrix answer it directly.

In [5]:
# ===== 4. Track correlations =====
net_cols = [c for c in daily.columns if c.endswith('_net') and (c.startswith('G10_') or c.startswith('ALL_'))]
corr = daily[net_cols].corr()
corr.to_csv(OUT / 'stage5_track_correlation.csv')

print('carry <-> momentum (net-track corr):')
for uni in ('G10', 'ALL'):
    for L in LOOKBACKS:
        print(f'  {uni}  {L:>3}d :', round(float(corr.loc[f'{uni}_carry_net', f'{uni}_mom{L}_net']), 3))

carry <-> momentum (net-track corr):
  G10   21d : -0.043
  G10   63d : -0.065
  G10  252d : 0.042
  ALL   21d : -0.005
  ALL   63d : 0.069
  ALL  252d : 0.139


## 5. Output + acceptance checks

In [6]:
# ===== 5. Write comparison CSV + acceptance asserts =====
summary.to_csv(OUT / 'stage5_momentum_comparison.csv')

strat_rows = [i for i in summary.index if i.startswith(('G10_', 'ALL_'))]
assert all(str(summary.loc[i, 'end']) == '2026-06-30' for i in strat_rows), 'window end'
# 252d-lookback books warm up ~4 months later; short/independent books start 2007-05
assert all(str(summary.loc[i, 'start']) <= '2007-09-30' for i in strat_rows), 'window start'
non_base = [i for i in strat_rows if '_carry_' not in i]
assert summary.loc[non_base, 't_alpha_vs_carry'].notna().all(), 'missing NW alpha'

# Reconciliation: vol-targeted pure carry == Stage-3 'voltgt'
s3 = pd.read_csv(OUT / 'stage3_dynamic_comparison.csv', index_col=0)
for uni in ('G10', 'ALL'):
    a = float(summary.loc[f'{uni}_carry_net', 'sharpe'])
    b = float(s3.loc[f'{uni}_voltgt_net', 'sharpe'])
    print(f'{uni} carry-net Sharpe {a:.3f}  vs Stage-3 voltgt {b:.3f}  (d={a-b:+.3f})')

# No-lookahead: momentum_panel on truncated data matches the full panel on overlap
cut = '2015-06-30'
mt = fx.momentum_panel(xret.loc[:cut], 63).dropna()
assert np.allclose(mt.iloc[-20:].values,
                   mom[63].reindex(mt.index).iloc[-20:].values, equal_nan=True), 'lookahead in momentum_panel'
print('no-lookahead + reconciliation checks passed;  rows =', len(strat_rows))

G10 carry-net Sharpe 0.119  vs Stage-3 voltgt 0.119  (d=+0.000)
ALL carry-net Sharpe 0.466  vs Stage-3 voltgt 0.466  (d=+0.000)
no-lookahead + reconciliation checks passed;  rows = 40


## 6. Verdict — does momentum beat the Stage-3 hedges?

**No.** Momentum does not reduce MaxDD / CVaR99 at *less* Sharpe cost than the
Stage-3 hedges — it costs far more Sharpe and, in the combined / filter cases,
*worsens* the drawdown. The per-currency-RR bar (combined net 0.466 → 0.457,
MaxDD −28%, skew −0.60) is untouched. Detail:

**Baseline** — pure carry (vol-targeted) reproduces Stage-3 `voltgt` exactly:
combined net Sharpe **0.466**, MaxDD −29.3%, CVaR99 2.9%, skew −0.65 (G10 0.119).
This is the reference every overlay is measured against.

**(a) Standalone momentum** *diversifies but does not pay.* Carry↔momentum net-track
correlation is near zero (−0.065 to +0.14), confirming the literature premise, and
standalone momentum flips skew positive (ALL 21d +0.09, G10 63d +1.37) and trims
CVaR99 (2.3–2.5% vs 2.9%). But every standalone-momentum track is a **net money
loser** (ALL net Sharpe −0.02 to −0.33; G10 −0.13 to −0.34) with deep drawdowns
(−0.52 to −0.73) — there is no investable FX momentum premium in this universe/window
net of costs. It can only ever enter as an overlay, not on its own.

**(b) Double-sort filter** *is dominated.* It cuts Sharpe (ALL net 0.24–0.37 vs
0.466) **and worsens the tail** (filter63 MaxDD −0.51 vs −0.29; CVaR99 3.0–3.4% vs
2.9%; skew no better). Filtering thins each leg, and the vol-target scalar then levers
the more concentrated book — the opposite of a hedge. `t_alpha_vs_carry` is
insignificant everywhere (|t| < 1.3). Reject.

**(c) Blend** *is also dominated.* 50/50 z-score blend gives up most of carry's
Sharpe (ALL net −0.02 to +0.16) while deepening MaxDD (−0.49 to −0.59). No lookback
helps in the combined book.

**Lookback robustness / mining guard.** The single apparent win — G10 blend @ 252d
(Sharpe 0.210 vs 0.119, MaxDD −0.22 vs −0.38, skew −0.18) — is **not robust**: it
appears only at 252d, only in G10 (G10 blend 21d/63d are negative), and vanishes in
the combined book (blend252 0.159 ≪ 0.466). Classic lookback-mining; **not adopted.**

**Bottom line.** Momentum's low carry correlation is real, but net of costs it
subtracts return faster than it buys tail protection. Per-currency RR remains the
preferred (near-free) tail hedge from Stage 3; momentum is not carried forward as a
standalone or overlay allocation. Its one durable use is as a **factor** — added to
the backtest §3 track regressions (see `strategy_backtest.ipynb` §3).

In [7]:
# ===== 6. Verdict comparison (combined book, net) =====
show = ['sharpe', 'max_drawdown', 'CVaR_99', 'skew', 'ann_cost_drag', 't_alpha_vs_carry']
def block(uni):
    rows = [f'{uni}_carry_net'] + [f'{uni}_{s}{L}_net'
            for s in ('mom', 'filter', 'blend') for L in LOOKBACKS]
    return summary.loc[rows, show].round(3)

print('=== COMBINED (ALL) — net ===')
print(block('ALL'))
print()
print('=== G10 — net ===')
print(block('G10'))
print()
carry_sh = float(summary.loc['ALL_carry_net', 'sharpe'])
carry_dd = float(summary.loc['ALL_carry_net', 'max_drawdown'])
carry_cv = float(summary.loc['ALL_carry_net', 'CVaR_99'])
carry_sk = float(summary.loc['ALL_carry_net', 'skew'])
print(f'pure-carry ALL net: Sharpe {carry_sh:.3f}  MaxDD {carry_dd:.3f}  CVaR99 {carry_cv:.3f}  skew {carry_sk:.3f}')
print('per-ccy RR bar:      Sharpe 0.457  MaxDD -0.28   skew -0.60 (Stage-3 tail-insurance winner)')

=== COMBINED (ALL) — net ===
                     sharpe max_drawdown   CVaR_99      skew ann_cost_drag  \
ALL_carry_net      0.465923    -0.293185  0.029199 -0.648005      0.018121   
ALL_mom21_net     -0.326792    -0.727796  0.023355  0.090758      0.046326   
ALL_mom63_net     -0.269104    -0.648518  0.025006 -0.114045      0.034525   
ALL_mom252_net    -0.019529    -0.521851  0.024577  -0.02973      0.024698   
ALL_filter21_net   0.284637    -0.373321  0.033344 -0.943561      0.027449   
ALL_filter63_net   0.372903    -0.505902  0.029736 -0.657734      0.021267   
ALL_filter252_net  0.241665    -0.440808  0.033817  -1.08636      0.017411   
ALL_blend21_net   -0.019293    -0.585424  0.025354  -0.34768      0.041302   
ALL_blend63_net     0.06933     -0.56007  0.025966  -0.40939      0.031833   
ALL_blend252_net   0.158997    -0.494857  0.027619 -0.369884      0.025196   

                  t_alpha_vs_carry  
ALL_carry_net                  NaN  
ALL_mom21_net            -1.506036  
A